In [ ]:
# Databricks Notebook: Bronze to Silver & Gold ETL for Climate Data (Scheduled)

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# 1. Initialize Spark session (already available in Databricks)
spark = SparkSession.builder.appName("ClimateDataETL").getOrCreate()

# 2. Load Bronze Table
bronze_df = spark.table("bronze.climatedata")

# Optional: Filter for recent data only (e.g., last 1 day)
bronze_df = bronze_df.filter(from_unixtime("last_updated") >= date_sub(current_date(), 1))


# 3. Create Silver Dimensions with MERGE (deduplication)

# dim_location
location_df = bronze_df.select(
    "country", "region", "location_name", "latitude", "longitude", "timezone"
).distinct()
location_df.createOrReplaceTempView("staging_location")
spark.sql("""
MERGE INTO silver.dim_location t
USING staging_location s
ON t.country = s.country AND t.region = s.region AND t.location_name = s.location_name
    AND t.latitude = s.latitude AND t.longitude = s.longitude AND t.timezone = s.timezone
WHEN NOT MATCHED THEN INSERT *
""")

# dim_time
time_df = bronze_df.select(
    to_date(from_unixtime("last_updated")).alias("full_date"),
    from_unixtime("last_updated").alias("full_timestamp"),
    year(from_unixtime("last_updated")).alias("year"),
    month(from_unixtime("last_updated")).alias("month"),
    dayofmonth(from_unixtime("last_updated")).alias("day"),
    hour(from_unixtime("last_updated")).alias("hour"),
    minute(from_unixtime("last_updated")).alias("minute"),
    date_format(from_unixtime("last_updated"), 'E').alias("weekday")
).distinct()
time_df.createOrReplaceTempView("staging_time")
spark.sql("""
MERGE INTO silver.dim_time t
USING staging_time s
ON t.full_date = s.full_date AND t.full_timestamp = s.full_timestamp
WHEN NOT MATCHED THEN INSERT *
""")

# dim_air_quality
air_df = bronze_df.select(
    col("air_quality_Carbon_Monoxide"),
    col("air_quality_Ozone"),
    col("air_quality_Nitrogen_dioxide"),
    col("air_quality_Sulphur_dioxide"),
    col("air_quality_PM2.5").alias("air_quality_PM2_5"),
    col("air_quality_PM10"),
    col("air_quality_us_epa_index"),
    col("air_quality_gb_defra_index")
).distinct()
air_df.createOrReplaceTempView("staging_air")
spark.sql("""
MERGE INTO silver.dim_air_quality t
USING staging_air s
ON t.air_quality_PM2_5 = s.air_quality_PM2_5 AND t.air_quality_PM10 = s.air_quality_PM10 AND
   t.air_quality_Carbon_Monoxide = s.air_quality_Carbon_Monoxide AND
   t.air_quality_Ozone = s.air_quality_Ozone AND
   t.air_quality_Nitrogen_dioxide = s.air_quality_Nitrogen_dioxide AND
   t.air_quality_Sulphur_dioxide = s.air_quality_Sulphur_dioxide AND
   t.air_quality_us_epa_index = s.air_quality_us_epa_index AND
   t.air_quality_gb_defra_index = s.air_quality_gb_defra_index
WHEN NOT MATCHED THEN INSERT *
""")

# dim_sky_condition
sky_df = bronze_df.select("condition_text", "cloud", "uv_index").distinct()
sky_df.createOrReplaceTempView("staging_sky")
spark.sql("""
MERGE INTO silver.dim_sky_condition t
USING staging_sky s
ON t.condition_text = s.condition_text AND t.cloud = s.cloud AND t.uv_index = s.uv_index
WHEN NOT MATCHED THEN INSERT *
""")

# dim_astronomy
astro_df = bronze_df.select("sunrise", "sunset", "moonrise", "moonset", "moon_phase", "moon_illumination").distinct()
astro_df.createOrReplaceTempView("staging_astro")
spark.sql("""
MERGE INTO silver.dim_astronomy t
USING staging_astro s
ON t.sunrise = s.sunrise AND t.sunset = s.sunset AND t.moonrise = s.moonrise AND
   t.moonset = s.moonset AND t.moon_phase = s.moon_phase AND t.moon_illumination = s.moon_illumination
WHEN NOT MATCHED THEN INSERT *
""")


# 4. Create Fact Table (INSERT only new data)


# Reload dimensions
dim_location = spark.table("silver.dim_location")
dim_time = spark.table("silver.dim_time")
dim_air_quality = spark.table("silver.dim_air_quality")
dim_sky_condition = spark.table("silver.dim_sky_condition")
dim_astronomy = spark.table("silver.dim_astronomy")

fact_df = bronze_df \
    .join(dim_location, on=["country", "region", "location_name", "latitude", "longitude", "timezone"], how="inner") \
    .join(dim_time, to_date(from_unixtime("last_updated")) == dim_time["full_date"], "inner") \
    .join(dim_air_quality,
          (bronze_df["air_quality_PM2.5"] == dim_air_quality["air_quality_PM2_5"]) &
          (bronze_df["air_quality_PM10"] == dim_air_quality["air_quality_PM10"]) &
          (bronze_df["air_quality_Carbon_Monoxide"] == dim_air_quality["air_quality_Carbon_Monoxide"]) &
          (bronze_df["air_quality_Ozone"] == dim_air_quality["air_quality_Ozone"]) &
          (bronze_df["air_quality_Nitrogen_dioxide"] == dim_air_quality["air_quality_Nitrogen_dioxide"]) &
          (bronze_df["air_quality_Sulphur_dioxide"] == dim_air_quality["air_quality_Sulphur_dioxide"]) &
          (bronze_df["air_quality_us_epa_index"] == dim_air_quality["air_quality_us_epa_index"]) &
          (bronze_df["air_quality_gb_defra_index"] == dim_air_quality["air_quality_gb_defra_index"]), "inner") \
    .join(dim_sky_condition,
          (bronze_df["condition_text"] == dim_sky_condition["condition_text"]) &
          (bronze_df["cloud"] == dim_sky_condition["cloud"]) &
          (bronze_df["uv_index"] == dim_sky_condition["uv_index"]), "inner") \
    .join(dim_astronomy,
          (bronze_df["sunrise"] == dim_astronomy["sunrise"]) &
          (bronze_df["sunset"] == dim_astronomy["sunset"]) &
          (bronze_df["moonrise"] == dim_astronomy["moonrise"]) &
          (bronze_df["moonset"] == dim_astronomy["moonset"]) &
          (bronze_df["moon_phase"] == dim_astronomy["moon_phase"]) &
          (bronze_df["moon_illumination"] == dim_astronomy["moon_illumination"]), "inner")

fact_final = fact_df.select(
    col("location_id"),
    col("time_id"),
    col("air_quality_id"),
    col("condition_id"),
    col("astronomy_id"),
    "temperature_celsius", "temperature_fahrenheit",
    "feels_like_celsius", "feels_like_fahrenheit",
    "humidity", "precip_mm", "precip_in", "wind_kph", "wind_mph",
    "wind_degree", "wind_direction", "pressure_mb", "pressure_in",
    "visibility_km", "visibility_miles", "gust_kph", "gust_mph"
)

fact_final.write.mode("append").saveAsTable("silver.fact_weather_observation")


# 5. Create or Replace Gold View

spark.sql("""
CREATE OR REPLACE VIEW gold.vw_weather_dashboard AS
SELECT
  f.observation_id,
  l.location_name, 
  l.region, 
  l.country,
  t.full_date,
  f.temperature_celsius,
  f.wind_mph,
  f.wind_kph,
  f.humidity,
  f.precip_mm,
  sc.condition_text,
  aq.air_quality_pm2_5,
  aq.air_quality_Carbon_Monoxide
  aq.air_quality_pm10,
  a.moon_phase
FROM silver.fact_weather_observation f
JOIN silver.dim_location l ON f.location_id = l.location_id
JOIN silver.dim_time t ON f.time_id = t.time_id
JOIN silver.dim_air_quality aq ON f.air_quality_id = aq.air_quality_id
JOIN silver.dim_sky_condition sc ON f.condition_id = sc.condition_id
JOIN silver.dim_astronomy a ON f.astronomy_id = a.astronomy_id
""")
	

print("ETL from Bronze to Silver and Gold completed successfully with deduplication and ready for scheduling.")
